In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import *

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DDMFA")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 09:02:00 WARN Utils: Your hostname, NOMAAN-ANV15, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 09:02:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/ddmfa/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nomaan/.ivy2.5.2/cache
The jars for the packages stored in: /home/nomaan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f86d9334-e181-4d5a-b1b3-bacc2fd25950;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central


In [3]:
from pathlib import Path

files = sorted(RAW_DATA_DIR.glob("*.csv"))

results = []

for file in files:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(file))
    )

    results.append(
        (
            file.name,
            df.count(),
            len(df.columns),
            ", ".join(df.columns)
        )
    )

result_df = spark.createDataFrame(
    results,
    ["file_name", "row_count", "column_count", "columns"]
)

result_df.orderBy("file_name").show(truncate=False)

[Stage 20:>                                                       (0 + 12) / 12]

+-----------------+---------+------------+----------------------------------------------------------------------------------------------+
|file_name        |row_count|column_count|columns                                                                                       |
+-----------------+---------+------------+----------------------------------------------------------------------------------------------+
|deliveries.csv   |5000     |8           |delivery_id, drone_id, source, destination, distance_km, start_time, end_time, status         |
|drones.csv       |100      |3           |drone_id, model, max_range_km                                                                 |
|drones_update.csv|5        |3           |drone_id, model, max_range_km                                                                 |
|flight_logs.csv  |30150    |8           |log_id, drone_id, delivery_id, timestamp, battery_level, gps_signal, weather_condition, status|
+-----------------+---------+-----

# Phase 2 — Silver Layer: Cleaning, Transformation & Failure Classification

**What this notebook does:**
- Reads all 3 Bronze Delta tables
- Casts columns to correct explicit types
- Fills nulls using median/mode strategies (recorded per column)
- Derives `delivery_duration_mins`, `failure_flag`, `failure_cause`
- Applies **threshold-based failure classification** on flight logs
- Runs **SCD Type 1 MERGE** on `silver_drones` using recalibrated specs
- Logs a Data Quality record to `dq_audit_log`
- Writes 3 Silver Delta tables

**Failure classification thresholds (design decision):**
- `battery_level < 20` → `FAILED_BATTERY`
- `gps_signal < 0.30` → `FAILED_SIGNAL`
- `weather_condition in [heavy_rain, storm]` → `FAILED_WEATHER`
- Priority: BATTERY > SIGNAL > WEATHER > UNKNOWN
- Classification is applied only where `status = FAILED`

In [4]:
from pyspark.sql.functions import (
    col, lit, when, current_timestamp,
    unix_timestamp, round as spark_round,
    to_timestamp
)
from pyspark.sql import DataFrame

# to track null fill operations across all silver tables
dq_log_rows = []

def null_fill_audit(df_before: DataFrame, df_after: DataFrame,
                    column: str, strategy: str, table: str):
    before_nulls = df_before.filter(col(column).isNull()).count()
    after_nulls  = df_after.filter(col(column).isNull()).count()
    filled       = before_nulls - after_nulls
    dq_log_rows.append((table, column, before_nulls, after_nulls, filled, strategy))
    print(f"  [{table}] {column}")
    print(f"    Nulls before : {before_nulls:,}")
    print(f"    Nulls after  : {after_nulls:,}")
    print(f"    Filled       : {filled:,}  (strategy: {strategy})")

## 1. `bronze_drones` → `silver_drones`
Type casting and metadata.
 
No nulls exist here (as confirmed in Bronze audit).

In [5]:
from pyspark.sql.functions import col

df_bronze_drones = (
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_drones"))
)

df_silver_drones = (
    df_bronze_drones
    .withColumn("drone_id", col("drone_id").cast("string"))
    .withColumn("model", col("model").cast("string"))
    .withColumn("max_range_km", col("max_range_km").cast("double"))
    .withColumn("processed_time", current_timestamp())
)

(
    df_silver_drones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(SILVER_DIR / "silver_drones"))
)

print(f"silver_drones written — {df_silver_drones.count():,} rows")
df_silver_drones.printSchema()

26/07/30 09:02:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

silver_drones written — 100 rows
root
 |-- drone_id: string (nullable = true)
 |-- model: string (nullable = true)
 |-- max_range_km: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- processed_time: timestamp (nullable = false)



## 2. SCD Type 1 MERGE on `silver_drones`
5 drones have recalibrated `max_range_km` values in `drones_update.csv`.

SCD Type 1 - overwrite the old value, no history kept.

For handling updates in the database.

In [6]:
from delta.tables import DeltaTable

# Read updates CSV
df_updates = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DRONE_UPDATES_FILE))
    .withColumn("processed_time", current_timestamp())
)

print("Incoming drone updates:")
df_updates.select("drone_id", "max_range_km").show(truncate=False)

updated_ids = [r["drone_id"] for r in df_updates.select("drone_id").collect()]

# Silver Delta table
silver_drones = DeltaTable.forPath(
    spark,
    str(SILVER_DIR / "silver_drones")
)

# BEFORE state
before_rows = (
    silver_drones.toDF()
    .filter(col("drone_id").isin(updated_ids))
    .select(
        "drone_id",
        col("max_range_km").alias("max_range_km_before")
    )
    .collect()
)

# MERGE (SCD Type 1)
(
    silver_drones.alias("target")
    .merge(
        df_updates.alias("source"),
        "target.drone_id = source.drone_id"
    )
    .whenMatchedUpdate(set={
        "max_range_km": "source.max_range_km",
        "processed_time": "source.processed_time"
    })
    .execute()
)

# AFTER state
df_after = (
    silver_drones.toDF()
    .filter(col("drone_id").isin(updated_ids))
    .select(
        "drone_id",
        col("max_range_km").alias("max_range_km_after")
    )
)

df_before = spark.createDataFrame(before_rows)

print("SCD Type 1 — Before vs After:")
df_before.join(df_after, "drone_id").show(truncate=False)

print("SCD Type 1 MERGE complete")

Incoming drone updates:
+--------+------------+
|drone_id|max_range_km|
+--------+------------+
|D038    |59.0        |
|D027    |110.1       |
|D079    |38.2        |
|D092    |47.1        |
|D050    |36.5        |
+--------+------------+



26/07/30 09:02:30 WARN MapPartitionsRDD: RDD 170 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


SCD Type 1 — Before vs After:
+--------+-------------------+------------------+
|drone_id|max_range_km_before|max_range_km_after|
+--------+-------------------+------------------+
|D027    |104.2              |110.1             |
|D038    |56.6               |59.0              |
|D050    |34.9               |36.5              |
|D079    |41.1               |38.2              |
|D092    |46.6               |47.1              |
+--------+-------------------+------------------+

SCD Type 1 MERGE complete


## 3. `bronze_deliveries` → `silver_deliveries`
- Fill `distance_km` nulls with **median** (50 nulls, 1.0%)
- Cast `start_time` and `end_time` to timestamp
- Derive `delivery_duration_mins` = (end_time - start_time) / 60
- Derive `failure_flag` = 1 if status = FAILED, else 0

In [7]:
df_bronze_del = (
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_deliveries"))
)

median_distance = df_bronze_del.approxQuantile("distance_km", [0.5], 0.01)[0]
print(f"Median distance_km: {median_distance:.2f} km (used to fill nulls)")

df_filled_del = df_bronze_del.fillna({"distance_km": round(median_distance, 2)})

print("\nNull Fill Audit:")
null_fill_audit(
    df_bronze_del,
    df_filled_del,
    "distance_km",
    f"median ({median_distance:.2f})",
    "silver_deliveries"
)

Median distance_km: 27.13 km (used to fill nulls)

Null Fill Audit:
  [silver_deliveries] distance_km
    Nulls before : 50
    Nulls after  : 0
    Filled       : 50  (strategy: median (27.13))


In [8]:
df_silver_del = (
    df_filled_del
    .withColumn("delivery_id",  col("delivery_id").cast("string"))
    .withColumn("drone_id",     col("drone_id").cast("string"))
    .withColumn("source",       col("source").cast("string"))
    .withColumn("destination",  col("destination").cast("string"))
    .withColumn("distance_km",  col("distance_km").cast("double"))
    .withColumn("start_time",   to_timestamp(col("start_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("end_time",     to_timestamp(col("end_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "delivery_duration_mins",
        spark_round(
            (unix_timestamp(col("end_time")) - unix_timestamp(col("start_time"))) / 60.0,
            2
        )
    )
    .withColumn(
        "failure_flag",
        when(col("status") == "FAILED", 1).otherwise(0)
    )
    .withColumn("processed_time", current_timestamp())
    .drop("ingestion_time", "source_file")
)

(
    df_silver_del.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(SILVER_DIR / "silver_deliveries"))
)

print(f"silver_deliveries written — {df_silver_del.count():,} rows")
df_silver_del.printSchema()

silver_deliveries written — 5,000 rows
root
 |-- delivery_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- distance_km: double (nullable = false)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- delivery_duration_mins: double (nullable = true)
 |-- failure_flag: integer (nullable = false)
 |-- processed_time: timestamp (nullable = false)



In [9]:
df_silver_del = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_deliveries"))
)

print("delivery_duration_mins distribution:")
df_silver_del.select("delivery_duration_mins").summary().show()

print("failure_flag distribution:")
df_silver_del.groupBy("failure_flag").count().show()

delivery_duration_mins distribution:
+-------+----------------------+
|summary|delivery_duration_mins|
+-------+----------------------+
|  count|                  5000|
|   mean|    27.859428000000015|
| stddev|     19.40884817783721|
|    min|                   1.8|
|    25%|                 13.18|
|    50%|                 23.32|
|    75%|                 37.62|
|    max|                141.67|
+-------+----------------------+

failure_flag distribution:
+------------+-----+
|failure_flag|count|
+------------+-----+
|           1|  907|
|           0| 4093|
+------------+-----+



## 4. `bronze_flight_logs` → `silver_flight_logs`
- Fill `battery_level` nulls with **median**
- Fill `gps_signal` nulls with **median**
- Fill `weather_condition` nulls with **mode**
- Derive `failure_flag` = 1 if status = FAILED, else 0
- Derive `failure_cause` using threshold-based classification

I fill Nulls BEFORE classification because a null battery level on a FAILED log
would make it unclassifiable without filling first.

In [10]:
df_bronze_logs = (
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_flight_logs"))
)

median_battery = df_bronze_logs.approxQuantile("battery_level", [0.5], 0.01)[0]
median_gps = df_bronze_logs.approxQuantile("gps_signal", [0.5], 0.01)[0]

mode_weather = (
    df_bronze_logs
    .filter(col("weather_condition").isNotNull())
    .groupBy("weather_condition")
    .count()
    .orderBy("count", ascending=False)
    .first()[0]
)

print(f"Median battery_level   : {median_battery:.2f}%")
print(f"Median gps_signal      : {median_gps:.3f}")
print(f"Mode weather_condition : {mode_weather}")

df_filled_logs = df_bronze_logs.fillna({
    "battery_level": round(median_battery, 2),
    "gps_signal": round(median_gps, 3),
    "weather_condition": mode_weather
})

print("\nNull Fill Audit:")
null_fill_audit(
    df_bronze_logs,
    df_filled_logs,
    "battery_level",
    f"median ({median_battery:.2f})",
    "silver_flight_logs"
)

null_fill_audit(
    df_bronze_logs,
    df_filled_logs,
    "gps_signal",
    f"median ({median_gps:.3f})",
    "silver_flight_logs"
)

null_fill_audit(
    df_bronze_logs,
    df_filled_logs,
    "weather_condition",
    f"mode ({mode_weather})",
    "silver_flight_logs"
)

Median battery_level   : 80.18%
Median gps_signal      : 0.767
Mode weather_condition : clear

Null Fill Audit:
  [silver_flight_logs] battery_level
    Nulls before : 875
    Nulls after  : 0
    Filled       : 875  (strategy: median (80.18))
  [silver_flight_logs] gps_signal
    Nulls before : 609
    Nulls after  : 0
    Filled       : 609  (strategy: median (0.767))
  [silver_flight_logs] weather_condition
    Nulls before : 568
    Nulls after  : 0
    Filled       : 568  (strategy: mode (clear))


In [11]:
df_silver_logs = (
    df_filled_logs
    .withColumn("log_id",            col("log_id").cast("string"))
    .withColumn("drone_id",          col("drone_id").cast("string"))
    .withColumn("delivery_id",       col("delivery_id").cast("string"))
    .withColumn("timestamp",         to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("battery_level",     col("battery_level").cast("double"))
    .withColumn("gps_signal",        col("gps_signal").cast("double"))
    .withColumn("weather_condition", col("weather_condition").cast("string"))
    .withColumn("status",            col("status").cast("string"))
    .withColumn(
        "failure_flag",
        when(col("status") == "FAILED", 1).otherwise(0)
    )
    .withColumn(
        "failure_cause",
        when(col("failure_flag") == 0, lit(None))
        .when(col("battery_level") < BATTERY_THRESHOLD, lit("FAILED_BATTERY"))
        .when(col("gps_signal") < SIGNAL_THRESHOLD, lit("FAILED_SIGNAL"))
        .when(col("weather_condition").isin(WEATHER_FAIL_CONDITIONS), lit("FAILED_WEATHER"))
        .otherwise(lit("UNKNOWN_FAILURE"))
    )
    .withColumn("processed_time", current_timestamp())
    .drop("ingestion_time", "source_file")
)

(
    df_silver_logs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(SILVER_DIR / "silver_flight_logs"))
)

print(f"silver_flight_logs written — {df_silver_logs.count():,} rows")
df_silver_logs.printSchema()

silver_flight_logs written — 30,000 rows
root
 |-- log_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- delivery_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- battery_level: double (nullable = false)
 |-- gps_signal: double (nullable = false)
 |-- weather_condition: string (nullable = false)
 |-- status: string (nullable = true)
 |-- failure_flag: integer (nullable = false)
 |-- failure_cause: string (nullable = true)
 |-- processed_time: timestamp (nullable = false)



## 5. Data Quality Audit Log

In [12]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

dq_schema = StructType([
    StructField("table_name",     StringType(),  True),
    StructField("column_name",    StringType(),  True),
    StructField("nulls_before",   IntegerType(), True),
    StructField("nulls_after",    IntegerType(), True),
    StructField("records_filled", IntegerType(), True),
    StructField("fill_strategy",  StringType(),  True),
])

df_dq_log = (
    spark.createDataFrame(dq_log_rows, schema=dq_schema)
    .withColumn("audit_time", current_timestamp())
)

(
    df_dq_log.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(SILVER_DIR / "dq_audit_log"))
)

print("Data Quality Audit Log:")
df_dq_log.show(truncate=False)
print("dq_audit_log written")

Data Quality Audit Log:
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+
|table_name        |column_name      |nulls_before|nulls_after|records_filled|fill_strategy |audit_time                |
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+
|silver_deliveries |distance_km      |50          |0          |50            |median (27.13)|2026-07-30 09:02:45.038287|
|silver_flight_logs|battery_level    |875         |0          |875           |median (80.18)|2026-07-30 09:02:45.038287|
|silver_flight_logs|gps_signal       |609         |0          |609           |median (0.767)|2026-07-30 09:02:45.038287|
|silver_flight_logs|weather_condition|568         |0          |568           |mode (clear)  |2026-07-30 09:02:45.038287|
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+

dq_audi

## 6. Validation

In [13]:
silver_tables = {
    "silver_drones": 100,
    "silver_deliveries": 5000,
    "silver_flight_logs": 30000,
}

print("Silver Layer Row Count Validation:")

for table, expected in silver_tables.items():
    actual = (
        spark.read
        .format("delta")
        .load(str(SILVER_DIR / table))
        .count()
    )

    status = "OK" if actual == expected else "CHECK"
    print(f"  {status}  {table:<25} {actual:>6,}  (expected {expected:,})")

Silver Layer Row Count Validation:
  OK  silver_drones                100  (expected 100)
  OK  silver_deliveries          5,000  (expected 5,000)
  OK  silver_flight_logs        30,000  (expected 30,000)


In [14]:
df_silver_logs = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_flight_logs"))
)

df_silver_del = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_deliveries"))
)

print("Failure Cause Distribution (silver_flight_logs — FAILED entries only):")
(
    df_silver_logs
    .filter(col("failure_flag") == 1)
    .groupBy("failure_cause")
    .count()
    .orderBy("count", ascending=False)
    .show()
)

total = df_silver_del.count()
failures = (
    df_silver_del
    .filter(col("failure_flag") == 1)
    .count()
)

print("Delivery Failure Rate:")
print(f"  Failed     : {failures:,}  ({failures/total*100:.1f}%)")
print(f"  Successful : {total - failures:,}  ({(total-failures)/total*100:.1f}%)")

Failure Cause Distribution (silver_flight_logs — FAILED entries only):
+---------------+-----+
|  failure_cause|count|
+---------------+-----+
| FAILED_BATTERY|  335|
|  FAILED_SIGNAL|  282|
|UNKNOWN_FAILURE|  188|
| FAILED_WEATHER|  102|
+---------------+-----+

Delivery Failure Rate:
  Failed     : 907  (18.1%)
  Successful : 4,093  (81.9%)


In [15]:
df_silver_drones = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_drones"))
)

df_silver_deliveries = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_deliveries"))
)

df_silver_logs = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "silver_flight_logs"))
)

print("silver_drones (3 rows):")
df_silver_drones.show(3, truncate=False)

print("silver_deliveries (3 rows):")
df_silver_deliveries.show(3, truncate=False)

print("silver_flight_logs — FAILED entries (5 rows):")
(
    df_silver_logs
    .filter(col("failure_flag") == 1)
    .select(
        "log_id",
        "drone_id",
        "delivery_id",
        "battery_level",
        "gps_signal",
        "weather_condition",
        "failure_flag",
        "failure_cause"
    )
    .show(5, truncate=False)
)

silver_drones (3 rows):
+--------+----------+------------+--------------------------+-----------+--------------------------+
|drone_id|model     |max_range_km|ingestion_time            |source_file|processed_time            |
+--------+----------+------------+--------------------------+-----------+--------------------------+
|D001    |Wing-G2   |57.4        |2026-07-30 09:01:06.867758|drones.csv |2026-07-30 09:02:19.765624|
|D002    |Skydio-D2 |75.5        |2026-07-30 09:01:06.867758|drones.csv |2026-07-30 09:02:19.765624|
|D003    |Zipline-R1|98.0        |2026-07-30 09:01:06.867758|drones.csv |2026-07-30 09:02:19.765624|
+--------+----------+------------+--------------------------+-----------+--------------------------+
only showing top 3 rows
silver_deliveries (3 rows):
+-----------+--------+---------------+-----------+-----------+-------------------+-------------------+-------+----------------------+------------+--------------------------+
|delivery_id|drone_id|source         |desti